In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir("/content/drive/MyDrive/Hotel Project")

# Load data
df = pd.read_csv("hotel_bookings.csv")

# Clean data
df["children"] = df["children"].fillna(0)
df["country"] = df["country"].fillna("Unknown")
df["agent"] = df["agent"].fillna(0)
df["company"] = df["company"].fillna(0)

df = df[~((df["adults"] == 0) & (df["children"] == 0) & (df["babies"] == 0))]

# Feature engineering
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

# Select features
features = [
    "hotel", "lead_time", "arrival_date_month", "total_nights",
    "total_guests", "meal", "country", "market_segment",
    "distribution_channel", "is_repeated_guest", "previous_cancellations",
    "booking_changes", "deposit_type", "customer_type", "adr",
    "required_car_parking_spaces", "total_of_special_requests"
]

X = df[features]
y = df["is_canceled"]

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Accuracy: 0.8788692223806728
              precision    recall  f1-score   support

           0       0.88      0.93      0.91     14958
           1       0.87      0.79      0.83      8884

    accuracy                           0.88     23842
   macro avg       0.88      0.86      0.87     23842
weighted avg       0.88      0.88      0.88     23842



In [6]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

results = X_test.copy()

results["actual_cancel"] = y_test
results["predicted_cancel"] = y_pred
results["cancel_probability"] = y_prob

In [7]:
def risk_level(p):
    if p < 0.3:
        return "Low Risk"
    elif p < 0.7:
        return "Medium Risk"
    else:
        return "High Risk"

results["risk_level"] = results["cancel_probability"].apply(risk_level)

In [8]:
results.to_csv("hotel_cancellation_predictions.csv", index=False)